<a href="https://colab.research.google.com/github/spandanakottur/sentiment-analysis/blob/hugging-face-dilbert-transformer/Hugging_Face_Finetuning_DIlbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install Dependencies**

Go to runtime and change it to GPU

In [1]:
import torch
torch.cuda.is_available()

True

In [2]:
!pip install datasets transformers huggingface_hub evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


**Pre process Data**

In [3]:
# Load data
from datasets import load_dataset
small_train_dataset = load_dataset("imdb",split="train")
small_train_dataset=small_train_dataset.shuffle().take(3000)
small_test_dataset=load_dataset("imdb",split="test")
small_test_dataset=small_train_dataset.shuffle().take(3000)
# We are taking a small portion of the dataset since a large dataset can slow things down,
# shuffle helps us randomize input before selcting 3000 entries for each dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [4]:
#Set up DistilBERT Tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
#Tokenization refers to breaking down input into smaller chunks called tokens.
#Tokens can be of any size they can be words or also characters,
# breaking it down makes it easier for machines to analyze
#Autotokenizer makes sure the relevant tokienizer class is created based on the input (model name) given as input

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [5]:
# Prepare the text inputs for the model
def preprocess_function(input):
    return tokenizer(input["text"], truncation=True)

tokenized_train = small_train_dataset.map(preprocess_function, batched=True)
tokenized_test = small_test_dataset.map(preprocess_function, batched=True)
#truncation parameter truncates the token to length specified by parameter max_length or maximimum length the model allows

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [6]:
# Use data_collector to convert our samples to PyTorch tensors and concatenate them with the correct amount of padding
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [7]:
# Define DistilBERT as our base model:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
import numpy as np
import evaluate

def compute_metrics(eval_pred):
  logits,labels=eval_pred
  load_accuracy=load_metric("accuracy")
  load_f1 = load_metric("f1")
  #f1 is a metric used to calculate how well the model does, usually when one class appears more frequently than the other, it uses precision and recall to do this
  #fl ensures that for a good score both precision and recall must be high




In [9]:
# Log in to your Hugging Face account
# Get your API token here https://huggingface.co/settings/token
from huggingface_hub import notebook_login

notebook_login()

In [18]:
#Define a new trainer
from transformers import TrainingArguments,Trainer

repo_name = "finetuning-sentiment-model-3000-samples"

training_args = TrainingArguments(
    output_dir=repo_name,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    save_strategy="epoch",
    push_to_hub=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# added extra argument report_to=None to avoid looging into wandb.ai
#learning rate is a hyperparameter that determnies the size of the steps taken to minimize the loss function in an optimization algorithm,
#if it is too large the model may overshoot and fail to converge, if it is too small it may take too long to converge
#Weight decay is a technique that keeps your machine learning model from becoming too complex and memorizing the training data instead of learning general rules

/tmp/ipython-input-1274987141.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [19]:
#train the model
trainer.train()

Step,Training Loss


TrainOutput(global_step=376, training_loss=0.29575918075886176, metrics={'train_runtime': 306.0075, 'train_samples_per_second': 19.607, 'train_steps_per_second': 1.229, 'total_flos': 785866982132928.0, 'train_loss': 0.29575918075886176, 'epoch': 2.0})

In [20]:
# Upload the model to the Hub
trainer.push_to_hub()

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...samples/training_args.bin: 100%|##########| 5.84kB / 5.84kB            

  ...51430.88a2ddcc5c68.1691.1: 100%|##########| 4.95kB / 4.95kB            

  ...51267.88a2ddcc5c68.1691.0: 100%|##########| 4.95kB / 4.95kB            

  ...samples/model.safetensors:   9%|9         | 25.1MB /  268MB            

CommitInfo(commit_url='https://huggingface.co/Twyla22/finetuning-sentiment-model-3000-samples/commit/cd323596982936d83cc7519ae90d5041bbf718c4', commit_message='End of training', commit_description='', oid='cd323596982936d83cc7519ae90d5041bbf718c4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Twyla22/finetuning-sentiment-model-3000-samples', endpoint='https://huggingface.co', repo_type='model', repo_id='Twyla22/finetuning-sentiment-model-3000-samples'), pr_revision=None, pr_num=None)

In [22]:
# Run inferences with your new model using Pipeline
from transformers import pipeline

sentiment_model = pipeline(model="federicopascual/finetuning-sentiment-model-3000-samples")

sentiment_model(["I had a weird day, I got my amazon Delivery today. I had a late lunch", "I had a weird day, I got my amazon Delivery today. I had a late lunch"])

Device set to use cuda:0


[{'label': 'LABEL_0', 'score': 0.7917524576187134},
 {'label': 'LABEL_0', 'score': 0.7917524576187134}]